In [1]:
import joblib
import os

MODEL_PATH = "xgboost_attrition_pipeline.pkl"

print("Checking model file...")

if os.path.exists(MODEL_PATH):
    print("Model file found.")
else:
    print("ERROR: Model file not found.")

Checking model file...
Model file found.


In [2]:
model = joblib.load(MODEL_PATH)

print("XGBoost model loaded successfully.")
print(type(model))

XGBoost model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


In [5]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import getpass

DB_USER = "postgres"
DB_PASSWORD = getpass.getpass("Enter PostgreSQL password: ")
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "workforce_intelligence"

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

print("Database engine created successfully.")

Enter PostgreSQL password:  ········


Database engine created successfully.


In [6]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Database connection successful:", result.fetchone())

Database connection successful: (1,)


In [7]:
import pandas as pd

workforce_df = pd.read_sql(
    "SELECT * FROM workforce_analytics",
    engine
)

print("Workforce data loaded successfully.")
print("Rows:", len(workforce_df))
print("Columns:", len(workforce_df.columns))

display(workforce_df.head())

Workforce data loaded successfully.
Rows: 1470
Columns: 33


,employee_id,age,gender,marital_status,education,education_field,department,job_level,job_role,business_travel,...,stock_option_level,environment_satisfaction,job_involvement,job_satisfaction,relationship_satisfaction,work_life_balance,performance_rating,training_times_last_year,attrition,attrition_flag
0,1,41,Female,Single,2,Life Sciences,Sales,2,Sales Executive,Travel_Rarely,...,0,2,3,4,1,1,3,0,Yes,1
1,2,49,Male,Married,1,Life Sciences,Research & Development,2,Research Scientist,Travel_Frequently,...,1,3,2,2,4,3,4,3,No,0
2,4,37,Male,Single,2,Other,Research & Development,1,Laboratory Technician,Travel_Rarely,...,0,4,2,3,2,3,3,3,Yes,1
3,5,33,Female,Married,4,Life Sciences,Research & Development,1,Research Scientist,Travel_Frequently,...,0,4,3,3,3,3,3,3,No,0
4,7,27,Male,Married,1,Medical,Research & Development,1,Laboratory Technician,Travel_Rarely,...,1,1,3,2,4,3,3,3,No,0


In [8]:
# Separate features from the actual target

X_prediction = workforce_df.drop(
    columns=["attrition_flag"],
    errors="ignore"
)

print("Prediction data prepared.")
print("Rows:", len(X_prediction))
print("Features:", len(X_prediction.columns))
print("Target column removed:", "attrition_flag" not in X_prediction.columns)

Prediction data prepared.
Rows: 1470
Features: 32
Target column removed: True


In [9]:
# Generate attrition predictions

predictions = model.predict(X_prediction)

# Probability that the employee will leave
prediction_probabilities = model.predict_proba(X_prediction)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(predictions))

Predictions generated successfully.
Number of predictions: 1470


In [10]:
prediction_results = workforce_df.copy()

# Add model predictions
prediction_results["predicted_attrition"] = predictions

# Add probability of leaving
prediction_results["attrition_probability"] = prediction_probabilities

# Convert 0/1 into readable labels
prediction_results["predicted_attrition_label"] = (
    prediction_results["predicted_attrition"]
    .map({
        0: "Stayed",
        1: "Left"
    })
)

print("Prediction results created successfully.")

display(
    prediction_results[
        [
            "employee_id",
            "predicted_attrition",
            "predicted_attrition_label",
            "attrition_probability"
        ]
    ].head(10)
)

Prediction results created successfully.


,employee_id,predicted_attrition,predicted_attrition_label,attrition_probability
0,1,1,Left,0.549069
1,2,0,Stayed,0.031508
2,4,1,Left,0.541161
3,5,0,Stayed,0.334790
4,7,0,Stayed,0.165226
5,8,0,Stayed,0.033857
6,10,0,Stayed,0.189110
7,11,0,Stayed,0.130677
8,12,0,Stayed,0.069097
9,13,0,Stayed,0.029447


In [11]:
def risk_category(probability):
    if probability < 0.30:
        return "Low Risk"
    elif probability < 0.60:
        return "Medium Risk"
    else:
        return "High Risk"


prediction_results["risk_category"] = (
    prediction_results["attrition_probability"]
    .apply(risk_category)
)

print("Risk categories created successfully.")

display(
    prediction_results[
        [
            "employee_id",
            "attrition_probability",
            "predicted_attrition_label",
            "risk_category"
        ]
    ].head(10)
)

Risk categories created successfully.


,employee_id,attrition_probability,predicted_attrition_label,risk_category
0,1,0.549069,Left,Medium Risk
1,2,0.031508,Stayed,Low Risk
2,4,0.541161,Left,Medium Risk
3,5,0.334790,Stayed,Medium Risk
4,7,0.165226,Stayed,Low Risk
5,8,0.033857,Stayed,Low Risk
6,10,0.189110,Stayed,Low Risk
7,11,0.130677,Stayed,Low Risk
8,12,0.069097,Stayed,Low Risk
9,13,0.029447,Stayed,Low Risk


In [12]:
print("Employee Risk Distribution:")
print(
    prediction_results["risk_category"]
    .value_counts()
)

Employee Risk Distribution:
risk_category
Low Risk       1245
Medium Risk     118
High Risk       107
Name: count, dtype: int64


In [13]:
print("\nRisk Distribution Percentage:")
print(
    (
        prediction_results["risk_category"]
        .value_counts(normalize=True) * 100
    ).round(2)
)


Risk Distribution Percentage:
risk_category
Low Risk       84.69
Medium Risk     8.03
High Risk       7.28
Name: proportion, dtype: float64


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Actual values from PostgreSQL
y_actual = workforce_df["attrition_flag"]

# Model predictions
y_predicted = prediction_results["predicted_attrition"]

print("===== PREDICTION VALIDATION =====")

print("Accuracy:",
      round(accuracy_score(y_actual, y_predicted), 4))

print("Precision:",
      round(precision_score(y_actual, y_predicted), 4))

print("Recall:",
      round(recall_score(y_actual, y_predicted), 4))

print("F1 Score:",
      round(f1_score(y_actual, y_predicted), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_predicted))

===== PREDICTION VALIDATION =====
Accuracy: 0.9293
Precision: 0.9586
Recall: 0.5865
F1 Score: 0.7277

Confusion Matrix:
[[1227    6]
 [  98  139]]


In [15]:
prediction_table = prediction_results[
    [
        "employee_id",
        "predicted_attrition",
        "predicted_attrition_label",
        "attrition_probability",
        "risk_category"
    ]
].copy()

print("Prediction table prepared.")
print("Rows:", len(prediction_table))

display(prediction_table.head(10))

Prediction table prepared.
Rows: 1470


,employee_id,predicted_attrition,predicted_attrition_label,attrition_probability,risk_category
0,1,1,Left,0.549069,Medium Risk
1,2,0,Stayed,0.031508,Low Risk
2,4,1,Left,0.541161,Medium Risk
3,5,0,Stayed,0.334790,Medium Risk
4,7,0,Stayed,0.165226,Low Risk
5,8,0,Stayed,0.033857,Low Risk
6,10,0,Stayed,0.189110,Low Risk
7,11,0,Stayed,0.130677,Low Risk
8,12,0,Stayed,0.069097,Low Risk
9,13,0,Stayed,0.029447,Low Risk


In [16]:
print("===== PREDICTED ATTRITION =====")

print(
    prediction_table["predicted_attrition_label"]
    .value_counts()
)

print("\n===== RISK CATEGORY =====")

print(
    prediction_table["risk_category"]
    .value_counts()
)

===== PREDICTED ATTRITION =====
predicted_attrition_label
Stayed    1325
Left       145
Name: count, dtype: int64

===== RISK CATEGORY =====
risk_category
Low Risk       1245
Medium Risk     118
High Risk       107
Name: count, dtype: int64


In [17]:
prediction_table.to_sql(
    "employee_attrition_predictions",
    engine,
    if_exists="replace",
    index=False
)

print("employee_attrition_predictions table created successfully.")

employee_attrition_predictions table created successfully.


In [18]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM employee_attrition_predictions;
        """)
    )

    count = result.scalar()

print("Prediction records in PostgreSQL:", count)

Prediction records in PostgreSQL: 1470


In [19]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT *
            FROM employee_attrition_predictions
            LIMIT 10;
        """)
    )

    rows = result.fetchall()

for row in rows:
    print(row)

(1, 1, 'Left', 0.54906875, 'Medium Risk')
(2, 0, 'Stayed', 0.031507988, 'Low Risk')
(4, 1, 'Left', 0.5411605, 'Medium Risk')
(5, 0, 'Stayed', 0.3347897, 'Medium Risk')
(7, 0, 'Stayed', 0.16522615, 'Low Risk')
(8, 0, 'Stayed', 0.033856746, 'Low Risk')
(10, 0, 'Stayed', 0.18910986, 'Low Risk')
(11, 0, 'Stayed', 0.13067716, 'Low Risk')
(12, 0, 'Stayed', 0.069097176, 'Low Risk')
(13, 0, 'Stayed', 0.029446587, 'Low Risk')


In [20]:
dashboard_view_query = """
CREATE OR REPLACE VIEW workforce_dashboard AS

SELECT
    w.employee_id,
    w.age,
    w.gender,
    w.marital_status,
    w.education,
    w.education_field,
    w.department,

    w.job_level,
    w.job_role,
    w.business_travel,
    w.total_working_years,
    w.years_at_company,
    w.years_in_current_role,
    w.years_since_last_promotion,
    w.years_with_curr_manager,
    w.num_companies_worked,
    w.distance_from_home,
    w.overtime,

    w.daily_rate,
    w.hourly_rate,
    w.monthly_income,
    w.monthly_rate,
    w.percent_salary_hike,
    w.stock_option_level,

    w.environment_satisfaction,
    w.job_involvement,
    w.job_satisfaction,
    w.relationship_satisfaction,
    w.work_life_balance,

    w.performance_rating,
    w.training_times_last_year,

    w.attrition,
    w.attrition_flag,

    p.predicted_attrition,
    p.predicted_attrition_label,
    p.attrition_probability,
    p.risk_category

FROM workforce_analytics w

LEFT JOIN employee_attrition_predictions p
    ON w.employee_id = p.employee_id;
"""

with engine.begin() as connection:
    connection.execute(text(dashboard_view_query))

print("workforce_dashboard view created successfully.")

workforce_dashboard view created successfully.


In [21]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM workforce_dashboard;
        """)
    )

    count = result.scalar()

print("Dashboard records:", count)

Dashboard records: 1470


In [22]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM workforce_dashboard
            WHERE predicted_attrition IS NULL;
        """)
    )

    missing_predictions = result.scalar()

print("Employees without predictions:", missing_predictions)

Employees without predictions: 0


In [23]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                employee_id,
                department,
                job_role,
                monthly_income,
                overtime,
                job_satisfaction,
                attrition,
                predicted_attrition_label,
                ROUND(
                    (attrition_probability * 100)::numeric,
                    2
                ) AS attrition_probability_percent,
                risk_category
            FROM workforce_dashboard
            LIMIT 10;
        """)
    )

    rows = result.fetchall()

for row in rows:
    print(row)

(1, 'Sales', 'Sales Executive', 5993, 'Yes', 4, 'Yes', 'Left', Decimal('54.91'), 'Medium Risk')
(2, 'Research & Development', 'Research Scientist', 5130, 'No', 2, 'No', 'Stayed', Decimal('3.15'), 'Low Risk')
(4, 'Research & Development', 'Laboratory Technician', 2090, 'Yes', 3, 'Yes', 'Left', Decimal('54.12'), 'Medium Risk')
(5, 'Research & Development', 'Research Scientist', 2909, 'Yes', 3, 'No', 'Stayed', Decimal('33.48'), 'Medium Risk')
(7, 'Research & Development', 'Laboratory Technician', 3468, 'No', 2, 'No', 'Stayed', Decimal('16.52'), 'Low Risk')
(8, 'Research & Development', 'Laboratory Technician', 3068, 'No', 4, 'No', 'Stayed', Decimal('3.39'), 'Low Risk')
(10, 'Research & Development', 'Laboratory Technician', 2670, 'Yes', 1, 'No', 'Stayed', Decimal('18.91'), 'Low Risk')
(11, 'Research & Development', 'Laboratory Technician', 2693, 'No', 3, 'No', 'Stayed', Decimal('13.07'), 'Low Risk')
(12, 'Research & Development', 'Manufacturing Director', 9526, 'No', 3, 'No', 'Stayed', De

In [25]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total_employees,
                COUNT(predicted_attrition_label) AS predicted_employees,
                COUNT(risk_category) AS risk_classified,
                ROUND(AVG(attrition_probability)::numeric, 2)
                    AS average_risk_probability
            FROM workforce_dashboard;
        """)
    )

    row = result.fetchone()

print("===== DASHBOARD DATA VALIDATION =====")
print("Total Employees:", row[0])
print("Employees with Predictions:", row[1])
print("Employees with Risk Category:", row[2])
print("Average Risk Probability:", row[3], "%")

===== DASHBOARD DATA VALIDATION =====
Total Employees: 1470
Employees with Predictions: 1470
Employees with Risk Category: 1470
Average Risk Probability: 0.16 %


In [26]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                risk_category,
                COUNT(*) AS employees
            FROM workforce_dashboard
            GROUP BY risk_category
            ORDER BY employees DESC;
        """)
    )

    rows = result.fetchall()

print("===== RISK CATEGORY SUMMARY =====")

for row in rows:
    print(f"{row[0]}: {row[1]} employees")

===== RISK CATEGORY SUMMARY =====
Low Risk: 1245 employees
Medium Risk: 118 employees
High Risk: 107 employees


In [28]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total_employees,
                COUNT(predicted_attrition_label) AS predicted_employees,
                COUNT(risk_category) AS risk_classified,
                ROUND(AVG(attrition_probability)::numeric, 4)
                    AS average_risk_probability
            FROM workforce_dashboard;
        """)
    )

    validation_row = result.fetchone()

print("===== DASHBOARD DATA VALIDATION =====")
print("Total Employees:", validation_row[0])
print("Employees with Predictions:", validation_row[1])
print("Employees with Risk Category:", validation_row[2])
print(
    "Average Risk Probability:",
    round(float(validation_row[3]) * 100, 2),
    "%"
)

===== DASHBOARD DATA VALIDATION =====
Total Employees: 1470
Employees with Predictions: 1470
Employees with Risk Category: 1470
Average Risk Probability: 15.88 %


In [29]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database();")
    )
    print("Database name:", result.fetchone()[0])

Database name: workforce_intelligence
